# 02 · Stage 1 screening: two-stage detectors (Faster R-CNN)


**Owner:** member B · **GPU:** ≈ 1.5 h (essential) / 4.5 h (all three) · **Needs:** 00 · **Parallel with:** 01, 03

Implementation: torchvision (no extra install, stable on Colab). Same protocol as 01 (1000 images, long side 960,
seed 42, fixed budget = 20 epochs, SGD, AMP, final checkpoint, unified evaluator).

| Config | Pretraining | Tier | Note |
|---|---|---|---|
| ResNet-50-FPN **v2** | COCO (torchvision v2 recipe, 46.7 COCO AP) | Essential | the "stronger/modern" configuration |
| ResNet-50-FPN v1 | COCO (37.0 COCO AP) | Important | classic reference |
| ResNet-101-FPN | ImageNet backbone + COCO neck/RPN/ROI heads transplanted from R50-FPN | Optional | torchvision ships no COCO R101-FPN; its result is **confounded by weaker pretraining** and must be reported as such |

Aerial adaptation used for all three (recorded in the spec): anchors shifted one octave down (16–256 px instead of 32–512)
because at input 960 most MAVREC objects are < 16 px; max 300 detections per image.

In [ ]:
# ===== 0. Setup — identical in every notebook. Runtime > Change runtime type > T4 GPU =====
SMOKE_TEST = False   # True: 16 train / 8 val images, 1 epoch — checks the whole pipeline in minutes
DRIVE_ROOT = "/content/drive/MyDrive/MAVREC"   # shared team folder (each member: "Add shortcut to Drive")

import os, sys, subprocess, importlib
try:
    from google.colab import drive
    try:
        os.listdir("/content/drive/MyDrive")   # raises if the Drive FUSE mount went stale
        drive.mount("/content/drive")
    except OSError:  # "Transport endpoint is not connected" after a Drive hiccup: remount
        drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not in Colab — make sure DRIVE_ROOT exists locally.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.173", "transformers==5.18.0", "pycocotools>=2.0.8", "timm>=1.0.15", "tabulate", "tensorboard", "seaborn"], check=True)

CODE_DIR = os.path.join(DRIVE_ROOT, "Final_Project_Code")
assert os.path.exists(os.path.join(CODE_DIR, "mavrec_lib.py")), "Run 00_Setup_and_Data_Preparation first (it installs mavrec_lib.py)."
sys.path.insert(0, CODE_DIR)
import mavrec_lib as ml; importlib.reload(ml)
CFG = ml.setup(DRIVE_ROOT, smoke=SMOKE_TEST or None)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
N_SMALL, N_SCREEN, N_LARGE = ml.TRAIN_SIZES      # 500, 1000, 2000

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment:
  python       : 3.13.15
  torch        : 2.11.0+cu130
  cuda         : True
  torchvision  : 0.26.0+cu130
  ultralytics  : 8.4.173
  transformers : 5.18.0
  pycocotools  : 2.0.11
  numpy        : 2.1.3
  lib          : 1.0.0
  gpu          : Tesla T4
  gpu_mem_gb   : 14.6


### S1 · Faster R-CNN R50-FPN v2 — **Essential**

In [ ]:
spec = ml.make_spec("frcnn_r50_v2", f"aerial_train_{N_SCREEN}", 960, stage="S1")
ml.run_experiment(spec)

Pack 'aerial_1280' unpacked to local disk in 46s
[S1_frcnn_r50_v2_n1000_r960_full_COCO_s42] training frcnn_r50_v2 | data=aerial_train_1000 | res=960 | epochs=20 | batch=4 | init=coco
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:00<00:00, 186MB/s]
/content/drive/MyDrive/MAVREC/Final_Project_Code/mavrec_lib.py:1817: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))


epoch 1/20 loss 1.6850 (137s)
epoch 2/20 loss 0.7396 (137s)
epoch 3/20 loss 0.5821 (138s)
epoch 4/20 loss 0.5029 (138s)
epoch 5/20 loss 0.4616 (138s)
epoch 6/20 loss 0.4256 (138s)
epoch 7/20 loss 0.3957 (138s)
epoch 8/20 loss 0.3754 (138s)
epoch 9/20 loss 0.3558 (137s)
epoch 10/20 loss 0.3437 (137s)
epoch 11/20 loss 0.3274 (137s)
epoch 12/20 loss 0.3148 (137s)
epoch 13/20 loss 0.3033 (137s)
epoch 14/20 loss 0.2959 (137s)
epoch 15/20 loss 0.2849 (137s)
epoch 16/20 loss 0.2782 (137s)
epoch 17/20 loss 0.2768 (137s)
epoch 18/20 loss 0.2718 (137s)
epoch 19/20 loss 0.2705 (137s)
epoch 20/20 loss 0.2684 (138s)
  predicted 538 images (full) in 43s, 91394 detections
[S1_frcnn_r50_v2_n1000_r960_full_COCO_s42] val|full|r960: mAP50-95=55.522 AP50=78.872 APS=36.051 APt=15.175 R=84.301 FPS=16.94


{'experiment_id': 'S1_frcnn_r50_v2_n1000_r960_full_COCO_s42',
 'stage': 'S1',
 'model_family': 'Two-stage',
 'model_version': 'frcnn_r50_v2',
 'backbone': 'ResNet-50-FPN v2',
 'dataset_size': 1000,
 'train_set': 'aerial_train_1000',
 'resolution': 960,
 'tiling': 'none',
 'pretraining': 'COCO',
 'init': 'coco',
 'epochs': 20,
 'seed': 42,
 'notes': '',
 'batch_size': 4,
 'train_time_h': 0.775,
 'GPU_memory_train_GB': 5.78,
 'checkpoint_path': '/content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S1_frcnn_r50_v2_n1000_r960_full_COCO_s42/final.pt',
 'spec': {'stage': 'S1',
  'model_version': 'frcnn_r50_v2',
  'model_family': 'Two-stage',
  'framework': 'torchvision',
  'backbone': 'ResNet-50-FPN v2',
  'train_set': 'aerial_train_1000',
  'val_set': 'aerial_val',
  'test_set': 'aerial_test',
  'train_pack': 'aerial_1280',
  'eval_pack': 'aerial_1280',
  'native_pack': 'aerial_native',
  'dataset_size': 1000,
  'resolution': 960,
  'train_res': 960,
  'epochs': 20,
  'batch_size': 4,
 

### S1 · Faster R-CNN R50-FPN v1 — **Important**

In [3]:
spec = ml.make_spec("frcnn_r50_v1", f"aerial_train_{N_SCREEN}", 960, stage="S1")
ml.run_experiment(spec)

[S1_frcnn_r50_v1_n1000_r960_full_COCO_s42] training frcnn_r50_v1 | data=aerial_train_1000 | res=960 | epochs=20 | batch=4 | init=coco
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 184MB/s]


epoch 1/20 loss 1.6009 (113s)
epoch 2/20 loss 0.9074 (112s)
epoch 3/20 loss 0.7412 (113s)
epoch 4/20 loss 0.6479 (113s)
epoch 5/20 loss 0.5929 (113s)
epoch 6/20 loss 0.5520 (113s)
epoch 7/20 loss 0.5095 (113s)
epoch 8/20 loss 0.4825 (113s)
epoch 9/20 loss 0.4638 (113s)
epoch 10/20 loss 0.4475 (113s)
epoch 11/20 loss 0.4261 (113s)
epoch 12/20 loss 0.4096 (113s)
epoch 13/20 loss 0.3981 (113s)
epoch 14/20 loss 0.3852 (114s)
epoch 15/20 loss 0.3771 (113s)
epoch 16/20 loss 0.3712 (115s)
epoch 17/20 loss 0.3656 (115s)
epoch 18/20 loss 0.3615 (115s)
epoch 19/20 loss 0.3592 (114s)
epoch 20/20 loss 0.3564 (114s)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s]


  predicted 538 images (full) in 40s, 115160 detections
[S1_frcnn_r50_v1_n1000_r960_full_COCO_s42] val|full|r960: mAP50-95=46.752 AP50=73.512 APS=25.735 APt=6.74 R=73.269 FPS=19.09


{'experiment_id': 'S1_frcnn_r50_v1_n1000_r960_full_COCO_s42',
 'stage': 'S1',
 'model_family': 'Two-stage',
 'model_version': 'frcnn_r50_v1',
 'backbone': 'ResNet-50-FPN',
 'dataset_size': 1000,
 'train_set': 'aerial_train_1000',
 'resolution': 960,
 'tiling': 'none',
 'pretraining': 'COCO',
 'init': 'coco',
 'epochs': 20,
 'seed': 42,
 'notes': '',
 'batch_size': 4,
 'train_time_h': 0.643,
 'GPU_memory_train_GB': 6.26,
 'checkpoint_path': '/content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S1_frcnn_r50_v1_n1000_r960_full_COCO_s42/final.pt',
 'spec': {'stage': 'S1',
  'model_version': 'frcnn_r50_v1',
  'model_family': 'Two-stage',
  'framework': 'torchvision',
  'backbone': 'ResNet-50-FPN',
  'train_set': 'aerial_train_1000',
  'val_set': 'aerial_val',
  'test_set': 'aerial_test',
  'train_pack': 'aerial_1280',
  'eval_pack': 'aerial_1280',
  'native_pack': 'aerial_native',
  'dataset_size': 1000,
  'resolution': 960,
  'train_res': 960,
  'epochs': 20,
  'batch_size': 4,
  'seed

### S1 · Faster R-CNN R101-FPN (pretraining-confounded) — **Optional**

In [4]:
RUN_R101 = False   # set True only if GPU time is left
if RUN_R101:
    spec = ml.make_spec("frcnn_r101", f"aerial_train_{N_SCREEN}", 960, stage="S1", notes="ImageNet R101 + transplanted COCO heads")
    ml.run_experiment(spec)

## Results (two-stage family)

In [5]:
df = ml.results_df(stage="S1")
t = df[df.model_family == "Two-stage"]
display(t[["experiment_id","stage","model_version","dataset_size","resolution","tiling","inference_mode","pretraining","mAP50_95","mAP50","APS","APt","APM","recall","F1","FPS","GPU_memory_infer_GB","parameters_M","train_time_h"]].sort_values("mAP50_95", ascending=False))
ml.save_table(t[["experiment_id","stage","model_version","dataset_size","resolution","tiling","inference_mode","pretraining","mAP50_95","mAP50","APS","APt","APM","recall","F1","FPS","GPU_memory_infer_GB","parameters_M","train_time_h"]], "S1_faster_rcnn")

,experiment_id,stage,model_version,dataset_size,resolution,tiling,inference_mode,pretraining,mAP50_95,mAP50,APS,APt,APM,recall,F1,FPS,GPU_memory_infer_GB,parameters_M,train_time_h
2,S1_frcnn_r50_v2_n1000_r960_full_COCO_s42,S1,frcnn_r50_v2,1000,960.0,none,full,COCO,55.522,78.872,36.051,15.175,55.217,84.301,88.288,16.94,0.43,43.30,0.775
1,S1_frcnn_r50_v1_n1000_r960_full_COCO_s42,S1,frcnn_r50_v1,1000,960.0,none,full,COCO,46.752,73.512,25.735,6.740,51.024,73.269,79.224,19.09,0.45,41.35,0.643


saved /content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S1_faster_rcnn.csv


'/content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S1_faster_rcnn'